# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an end-to-end workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant JSON-LD schema URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Keep as object, not dict
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets and fields by `@id`. Inspect dataset structure to plan data extraction and EDA.

In [ ]:
# Retrieve record set `@id`s
record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in getattr(metadata, 'recordSet', [])]
print("Available Record Set @id's:")
for rs_id in record_sets:
    print(f"- {rs_id}")

# Show first record and its field structure for each record set
for rs_id in record_sets:
    print(f"\nFields for record set {rs_id}:")
    try:
        # fields() will list field metadata, records() gives full records
        fields = dataset.fields(record_set=rs_id)
        for field in fields:
            print(f"    Field @id: {field['@id']}, name: {field.get('name','')}  (type: {field.get('dataType','')})")
        # Print a preview record
        first_records = list(dataset.records(record_set=rs_id))[:1]
        print(f"    Sample record: {first_records[0] if first_records else '{}'}")
    except Exception as e:
        print(f"    Error loading {rs_id}: {e}")

## 3. Data Extraction
Load data from record sets into DataFrames. Use field `@id`s for all manipulations as recommended.

In [ ]:
# List of record set @ids (use overview section outputs)
record_set_ids = record_sets
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading data from record set: {rs_id}")
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample rows:\n{df.head()}\n")
    except Exception as e:
        print(f"  Error loading records for {rs_id}: {e}")

# Pick a record set for further analysis (choose first for demonstration)
main_record_set = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set:
    print(f"Using record set: {main_record_set} for EDA")
    print(f"Columns in this set: {dataframes[main_record_set].columns.tolist()}")
    dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering by a numeric field, normalization, grouping. Use only field and record set `@id`s.

In [ ]:
# Identify candidate numeric and group fields via prior cells
# Example: Assume field @ids 'cr:protein_coverage' (numeric) and 'cr:sample_id' (group). Replace according to real '@id's.
df = dataframes[main_record_set]
numeric_candidates = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[c])]
group_candidates = [c for c in df.columns if 'sample' in c.lower() or 'group' in c.lower()]

# Assign sensible defaults if possible
numeric_field_id = numeric_candidates[0] if numeric_candidates else None
group_field_id = group_candidates[0] if group_candidates else None
print(f"Numeric field (for filtering): {numeric_field_id}")
print(f"Group field: {group_field_id}")

if numeric_field_id:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {filtered_df.shape[0]} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field if present
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped means by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of a selected numeric field and relationship to group(s), if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If grouping possible, boxplot per group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated programmatic data extraction, EDA, and visualization on the FAIR^2 mass spectrometry dataset using `mlcroissant`. Patterns and anomalies in protein-related fields can be further explored for biological insight. For production analysis, replace hardcoded field IDs with those found dynamically, and always reference fields and record sets by their Croissant `@id`.